# Sampling a timing model with `nltiming`

In a typical PTA analysis you **do not sample** the timing model. You fix the
par-file values and analytically marginalize a linear timing design matrix
(Enterprise `TimingModel`, Discovery improper GP, …). That is the right default
when the parameters of interest are noise or a GW signal and the timing model
is only a nuisance — and when the likelihood is close enough to linear in those
timing parameters (it is not always).

Sometimes you *do* need the timing parameters in the posterior — binary
parameters for a CW search, dispersion when it correlates with the signal,
full-basis “decentering” where every fitpar is sampled jointly with the noise.
`nltiming` is the library for that: it plugs a **nonlinear timing likelihood**
into Discovery or Enterprise and lets you choose, per fitpar, whether to
**sample** it or **analytically marginalize** it.

**Pulsar object (today):** `nltiming` talks to pulsars through a small
`TimingPulsar` protocol (`pint_model()`, `timing_engine()`, frozen TOA arrays,
…). Right now the only production implementation of that contract is
[MetaPulsar](https://github.com/vhaasteren/metapulsar) — even for a single PTA
dataset — so these notebooks build the pulsar with `create_metapulsar`. Once
Discovery and/or Enterprise grow a native `TimingPulsar`, that dependency can
be dropped; the `nltiming` calls below will not change.

This notebook assumes you know tempo2/PINT and can already sample a
deterministic (e.g. CW-like) model in Discovery/Enterprise. It does **not**
assume you have sampled timing parameters before. We will cover, from the ground
up:

1. The **inference plan** — which axes are sampled, which are marginalized, and
   *how* (two different analytical measures).
2. **Coordinate charts** — why the sampler does not walk in raw physical units,
   and how each axis maps physical offset ↔ prior-normal coordinate.
3. A short **joint NUTS** run that samples every timing axis together with a
   (fixed) red-noise model, then decodes draws back to physical values.
4. The same plan through **Enterprise**.

Simulated PINT data; no external dataset. Run top-to-bottom in an environment
with MetaPulsar, JUG, Discovery, and NumPyro (e.g. the MetaPulsar devcontainer).


In [1]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import tempfile
from pathlib import Path

import jax
import numpy as np

import discovery as ds
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import NonLinearTimingModel, TimingInference, WhiteningConfig
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")
workdir = Path(tempfile.mkdtemp(prefix="nlt_charts_"))


## 1. A simulated pulsar

~150 TOAs from PINT’s bundled `NGC6440E` example. The TOAs are drawn from the
timing model plus white noise, so we know the true parameters exactly — useful
later when we decode NUTS samples.


In [2]:
import pint.config
from pint.models import get_model
from pint.simulation import make_fake_toas_uniform

np.random.seed(42)
model = get_model(pint.config.examplefile("NGC6440E.par"))
toas = make_fake_toas_uniform(
    startMJD=53400, endMJD=56000, ntoas=150, model=model, obs="gbt",
    error=1.0, add_noise=True,
)
par_path, tim_path = workdir / "demo.par", workdir / "demo.tim"
par_path.write_text(model.as_parfile())
toas.write_TOA_file(str(tim_path), format="tempo2")

# MetaPulsar is required today: it is the only TimingPulsar implementation
# nltiming can bind to (even for one PTA). Native Discovery/Enterprise
# TimingPulsar support will remove this wrapper; the nltiming API stays the same.
mp = create_metapulsar(
    {"demo": [{"par": str(par_path), "tim": str(tim_path), "timing_package": "pint"}]},
    use_pulse_numbers="no",
)
print("pulsar :", mp.name, " TOAs:", len(mp.toas))
print("fitpars:", list(mp.fitpars))


pulsar : J1748-2021  TOAs: 150
fitpars: ['RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo']


## 2. The inference plan: sample or analytically marginalize?

Every fitpar gets exactly one **disposition**:

| Disposition | What happens |
|---|---|
| `sample` | The exact timing engine evaluates that parameter; NUTS (or PTMCMC) samples it. |
| `marginalize_delta_flat` | Classic PTA linear marginalization: improper flat prior on the physical offset δ. |
| `marginalize_z_prior` | Linearize in a prior-whitened coordinate `z` and integrate a *proper* unit-normal prior. |

The last two are both “analytical marginalization,” but they are **different
scientific models** (different measure → different evidence → different run
fingerprint). You name what is *marginalized*; everything else is sampled.

Everyday presets on `NonLinearTimingModel`:

- omit `inference`, or pass `inference="default"` — marginalize the usual linear
  nuisances (spindown, DM, jumps, …) as delta-flat; sample the rest.
- `inference="all"` — sample every timing axis (needed for joint full-basis /
  decentering analyses).
- `TimingInference.groups(delta_flat=[...], z_prior=[...])` — pick axes and
  measures by hand.

`for_pulsar(mp)` resolves the plan against this pulsar’s fitpars (including
PTA-suffixed names). Printing the model or `ctx.plan` shows the result.


In [3]:
for inf in [
    "default",
    TimingInference.groups(z_prior=["DM"], delta_flat=["DM1", "DM2"]),
    "all",
]:
    c = NonLinearTimingModel(engines="jug", inference=inf, name="timing").for_pulsar(mp)
    print(c.model.inference)
    print(f"  {c.plan}")


[!] Unrecognized par file parameters (ignored by JUG): DMDATA, NE_SW1
[FITTER] noise_config was None, auto-detecting from par file


/workspaces/metapulsar/ref-packages/nltiming/src/nltiming/nonlinear_timing_model.py:1304: LocallyMarginalizedTimingWarning: analytically marginalized timing axes are not certified identically linear, so their integration is local: ['RAJ', 'DECJ', 'F0', 'F1']. The plan is honored; pass coordinate_policy with nonidentically_linear_marginalization='ignore' to silence.
  base = self._unconditioned_for_pulsar(pulsar, engine)


default()                                        sampled=()
                                                 marg_delta=('RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo') marg_z=()
groups(z_prior=[DM], delta_flat=[DM1, DM2])      sampled=('RAJ', 'DECJ', 'F0', 'F1', 'Offset_demo')
                                                 marg_delta=('DM1', 'DM2') marg_z=('DM',)
sample_all()                                     sampled=('RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo')
                                                 marg_delta=() marg_z=()


## 3. Why “charts”? Sampling in a whitened coordinate

Timing parameters differ wildly in scale (`F0` vs `DM2`) and are highly
covariant. Sampling them in raw physical units is a bad idea — the same reason
you do not sample a CW amplitude and frequency on mismatched raw scales without
care.

`nltiming` works with a physical offset **δ** from the par-file reference, then
maps each axis through a **coordinate chart** to a prior-normal coordinate **z**
(standard normal under the physical prior):

```
δ  (physical offset)  ←── chart ──→  z  (prior-normal)  ←── optional affine layer ──→  ξ (sampler)
```

### What “chart” means

In ordinary statistical language this is a **prior-normalizing reparameterization**
(or bijector / parameter transform). We also use the geometric word **chart**
because it names a useful distinction:

- A **chart** maps a physical parameter point to a coordinate representation
  (δ → z).
- A subsequent **transport** maps between coordinate representations chosen for
  inference (z ↔ ξ; identity or static whitening, or the dynamic joint map).

Formally, a coordinate chart on a manifold is an invertible map from an open
region of the underlying space onto coordinates in Euclidean space (Lee 2013,
Ch. 1). Here each timing axis is a one-dimensional interval or line; the chart
is the chosen map that puts that axis into the `z` coordinate the rest of the
stack uses for priors, Jacobians, and whitening. We do not lean on a full
manifold atlas beyond that — “chart” emphasizes that δ is the physical point
and z is merely its coordinate.

Keep three objects distinct (easy to conflate):

```
δ ↦ z            exact prior reparameterization (the chart)
Dδ/Dz |_{z_e}    local Jacobian at the expansion point
z = c + C ξ      local affine whitening (static layer)
```

The chart itself is a full (usually nonlinear) bijection over the prior support —
not a tangent-plane approximation. The Jacobian and the static whitening layer
built from it *are* local. Hyperparameter dependence in the dynamic joint
transport makes that transport conditional on η; it does not make the per-axis
charts any less exact.

Practically:

1. **One prior ⇒ one chart.** The chart is derived from the physical prior on δ;
   you do not pick an independent transform that would imply a different
   measure.
2. **Charts ≠ the whole stack.** Only the per-axis `δ ↔ z` map is a chart. The
   optional shared affine map `z → ξ` (identity or static whitening) is a
   separate *layer*, not another chart.
3. **Exact prior map vs local whitening.** `affine_normal` is globally affine;
   `prior_pit` is exact throughout the interior of the prior support, while a
   static whitening layer from its Jacobian at one expansion point is only a
   local approximation to posterior geometry (see notebook **02**).

**Reference.** J. M. Lee, *Introduction to Smooth Manifolds*, 2nd ed., Graduate
Texts in Mathematics 218, Springer (2013), Chapter 1 (“Smooth Manifolds”) —
definition of a coordinate chart (or just *chart*) on a topological manifold.
DOI: [10.1007/978-1-4419-9982-5](https://doi.org/10.1007/978-1-4419-9982-5).

### Which chart you get

For joint full-basis NUTS we keep the static layer as the identity
(`whitening=None`), so the sampler coordinate is `z` itself. Which chart an
axis gets depends on the physical prior on δ:

- **Gaussian δ prior** → globally affine `affine_normal` chart (typical for axes
  declared *identically linear* in the timing delay).
- **Bounded / non-Gaussian prior** (e.g. wide uniform “cheat” prior) → nonlinear
  **PIT** chart (`prior_pit`: probability integral transform through the prior
  CDF and the standard-normal quantile function). Exact throughout the interior
  of the prior support; only the static whitening built from its Jacobian at
  the expansion point is a local approximation.

`ctx.chart_summary()` lists disposition, chart, identical-linearity flag, and
resolved prior for every proper axis (sampled or z-prior marginalized).


In [4]:
ctx = NonLinearTimingModel(
    engines="jug", inference="all", name="timing"
).for_pulsar(mp)

print(f"{'axis':14s} {'disposition':14s} {'chart':14s} {'lin?':5s} prior")
for d in ctx.chart_summary():
    print(f"{d['name']:14s} {d['disposition']:14s} {d['chart']:14s} "
          f"{str(d['identically_linear']):5s} {d['prior_family']} ({d['prior_source']})")
print("\nidentically linear:", ctx.identically_linear)


axis           disposition    chart          lin?  prior
RAJ            sample         prior_pit      False uniform (cheat_wls)
DECJ           sample         prior_pit      False uniform (cheat_wls)
F0             sample         prior_pit      False uniform (cheat_wls)
F1             sample         prior_pit      False uniform (cheat_wls)
DM             sample         affine_normal  True  normal (cheat_wls)
DM1            sample         affine_normal  True  normal (cheat_wls)
DM2            sample         affine_normal  True  normal (cheat_wls)
Offset_demo    sample         affine_normal  True  normal (cheat_wls)

identically linear: ('DM', 'DM1', 'DM2', 'Offset_demo')


## 4. Two ways to analytically marginalize

If you come from Enterprise/Discovery, “marginalize the timing model” almost
always means an **improper flat** measure on the linear coefficients
(delta-flat). `nltiming` also offers a **proper** unit-normal measure in the
whitened coordinate (z-prior). Same named axes, different evidence — the plan
fingerprint records which you chose so run products cannot be confused.


In [5]:
subset = ["DM"]
ctx_df = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(delta_flat=subset), name="timing"
).for_pulsar(mp)
ctx_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset), name="timing"
).for_pulsar(mp)
print("delta-flat plan:", ctx_df.plan.fingerprint()[:24])
print("z-prior    plan:", ctx_zp.plan.fingerprint()[:24])
print("distinct        :", ctx_df.plan.fingerprint() != ctx_zp.plan.fingerprint())


delta-flat plan: sha256:167bbaec6514c9917
z-prior    plan: sha256:488f67a2e76809955
distinct        : True


## 5. Assemble a Discovery likelihood and run NUTS

The pulsar context exposes Discovery signals the same way a CW or red-noise
component would. With `inference="all"` every timing axis is sampled; we build a
**joint** residual-form likelihood (`joint=True`) so timing and GP coefficients
share one latent vector.

`N.joint_model` wraps that likelihood for NumPyro. `N.nuts` is ordinary NUTS with
sensible defaults: `dense_mass="auto"` adapts a dense mass block on red-noise
hyperparameters (when free) while leaving the intended-white timing latent on an
identity mass. Here we **fix** the noise for a fast timing-only demo — free
them for science.


In [6]:
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
psl = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 15, name="rednoise"),
    *ctx.discovery_signals(joint=True),
])
# Fix the red-noise hyper for a fast timing-only demo (leave them free for science).
fixed = {**nd,
         f"{mp.name}_rednoise_log10_A": -14.0,
         f"{mp.name}_rednoise_gamma": 3.0}
jm = N.joint_model(psl, ctx, fixed=fixed)
print("xi site:", jm.xi_site, " dim:", jm.transport.dimension, " hyper:", jm.hyper_sites)

mcmc = N.nuts(jm, ctx, num_warmup=500, num_samples=1000, progress_bar=True)
mcmc.run(jax.random.PRNGKey(0), extra_fields=N.NUTS_EXTRA_FIELDS)


xi site: J1748-2021_timing_joint_xi  dim: 38  hyper: ()


sample: 100%|██████████| 1500/1500 [00:09<00:00, 150.66it/s, 127 steps of size 3.03e-02. acc. prob=0.71]


### Decode samples to physical timing parameters

The chain stores draws in the sampler’s latent coordinates, not in Hz or
pc cm⁻³. `jm.to_df` applies the transport and each axis’s chart so you get
physical columns (here compared to the simulation truth).


In [7]:
df = jm.to_df(mcmc.get_samples())
for name in ("F0", "F1"):
    col = np.asarray(df[f"{ctx.name_stem}_{name}_theta_native"])
    truth = float(getattr(model, name).value)
    lo, hi = np.percentile(col, [16, 84])
    print(f"{name}: truth={truth:.12g}  68%=[{lo:.12g}, {hi:.12g}]")


F0: truth=61.485476554  68%=[61.4854765524, 61.4854765537]
F1: truth=-1.181e-15  68%=[-1.18394331227e-15, -1.1745253638e-15]


## 6. Look at each chain before pooling

Same discipline as a CW run: diagnose chains separately. `N.chain_diagnostics`
keeps `group_by_chain=True` samples and NUTS integrator fields (tree depth,
accept probability). Pass `extra_fields=N.NUTS_EXTRA_FIELDS` to `mcmc.run` so
those fields are recorded.


In [8]:
diag = N.chain_diagnostics(mcmc, max_tree_depth=10)
print("num_steps shape (chains, draws):", diag["num_steps"].shape)
print("tree-depth saturation fraction :", round(diag["tree_depth_saturation_fraction"], 4))
print("saturated (>10%)               :", diag["tree_depth_saturated"])
print("mean accept prob               :", round(float(diag["accept_prob"].mean()), 3))


num_steps shape (chains, draws): (1, 1000)
tree-depth saturation fraction : 0.0
saturated (>10%)               : False
mean accept prob               : 0.714


## 7. Same plan through Enterprise

If you build PTAs in Enterprise rather than Discovery, call
`ntm.enterprise_signal()` — same inference plan, Enterprise-native parameters.
With `sample_z_coefficients=True`, a z-prior marginalized block is promoted from
an integrated GP to sampled unit-normal coefficients (useful when you want those
coefficients in a joint/decentered vector instead of integrating them).


In [9]:
from enterprise.signals import parameter, signal_base, white_signals

ntm_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=["DM"]),
    whitening=WhiteningConfig(), name="timing",
)
white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
pta = signal_base.PTA([(white + ntm_zp.enterprise_signal(sample_z_coefficients=True))(mp)])
print("enterprise params:", pta.param_names)


enterprise params: ['J1748-2021_timing_x_0', 'J1748-2021_timing_x_1', 'J1748-2021_timing_x_2', 'J1748-2021_timing_x_3', 'J1748-2021_timing_x_4', 'J1748-2021_timing_x_5', 'J1748-2021_timing_x_6', 'J1748-2021_timing_zprior_coefficients_0']
